In [1]:
from datasets import load_dataset

dataset = load_dataset("wandb/RAGTruth-processed")

Repo card metadata block was not found. Setting CardData to empty.


In [2]:
import pandas as pd

from hallucination_detector.data import clean, split_stratified

df = pd.DataFrame(dataset["train"])
clean_df = clean(df)
train, val = split_stratified(clean_df)

In [3]:
from hallucination_detector.baselines import MajorityBaseline

model = MajorityBaseline()
model.fit(train["label"].tolist())

y_true = val["label"].tolist()
y_pred = [model.predict(x) for x in val["output"]]
y_score = [model.score(x) for x in val["output"]]

In [4]:
from hallucination_detector.metrics import (
    f1,
    fbeta,
    mean_latency_ms,
    precision,
    recall,
    roc_auc,
)

accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
print("accuracy", accuracy)
print("precision", precision(y_true, y_pred))
print("recall", recall(y_true, y_pred))
print("f1", f1(y_true, y_pred))
print("fbeta", fbeta(y_true, y_pred, beta=2.0))
print("roc_auc", roc_auc(y_true, y_score))
print("mean_latency_ms", mean_latency_ms(model.predict, val["output"].tolist()))

accuracy 0.5540780141843972
precision 0.0
recall 0.0
f1 0.0
fbeta 0.0
roc_auc 0.5
mean_latency_ms 9.06028386059348e-05


C:\Users\maham\Personal Projects\HallucinationDetector\src\hallucination_detector\metrics.py:29: UserWarning: No positive predictions; precision set to 0.0
  warnings.warn("No positive predictions; precision set to 0.0")


In [5]:
from hallucination_detector.baselines import WordOverlapBaseline

model_1 = WordOverlapBaseline()
X_train = list(zip(train["output"], train["context"]))

model_1.fit(X_train, train["label"].tolist())

X_val = list(zip(val["output"], val["context"]))
y_true_o = val["label"].tolist()
y_pred_o = [model_1.predict(o) for o in X_val]
y_score_o = [model_1.score(o) for o in X_val]

In [6]:
accuracy = sum(t == p for t, p in zip(y_true_o, y_pred_o)) / len(y_true_o)
print("accuracy", accuracy)
print("precision", precision(y_true_o, y_pred_o))
print("recall", recall(y_true_o, y_pred_o))
print("f1", f1(y_true_o, y_pred_o))
print("fbeta", fbeta(y_true_o, y_pred_o, beta=2.0))
print("roc_auc", roc_auc(y_true_o, y_score_o))
print("mean_latency_ms", mean_latency_ms(model_1.predict, X_val))

accuracy 0.5602836879432624
precision 0.5036307053941909
recall 0.9652087475149106
f1 0.6618950238582141
fbeta 0.8156922043010754
roc_auc 0.7383745526838966
mean_latency_ms 0.1493074024891236
